# Fluorescence Anisotropy Binding Analysis and Model Selection
# Author of script : Dr. Jaydeep Paul
# For Bug report, contact: jaydeepccmb@gmail.com , jaydeepbm2paul@gmail.com
# Free to use and modifications
This notebook fits fluorescence anisotropy titration data to multiple binding models, performs statistical model selection using **AICc** (Akaike Information Criterion corrected for small sample size), and estimates parameter uncertainties using **Monte Carlo** simulations.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import seaborn as sns
import os


##  Load Experimental Data
 file-loader block that uses fallback synthetic data if the file is not yet available.

In [2]:
file_path = "test_aniso.xlsx"

if os.path.exists(file_path):
    df = pd.read_excel(file_path)
    x_data = df['ligand'].values
    y_data = df['anisotropy'].values
    print(f"Loaded experimental data from '{file_path}':")
    print(df.head())
else:
    print(f"'{file_path}' not found. Using representative fluorescence anisotropy dataset:")
    # Standard representative titration data (Ligand vs Anisotropy in mP)
    x_data = np.array([0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 250.0, 500.0])
    y_data = np.array([60.0, 68.0, 85.0, 115.0, 150.0, 175.0, 185.0, 205.0, 240.0, 275.0, 295.0, 305.0])
    
    df_demo = pd.DataFrame({'ligand': x_data, 'anisotropy': y_data})
    print(df_demo)

Loaded experimental data from 'test_aniso.xlsx':
   ligand  anisotropy
0   0.000    0.041961
1   0.975    0.042113
2   1.950    0.043033
3   3.900    0.043801
4   7.810    0.043646


## 1. Define Binding Models
We define 5 potential binding models, incorporating corrections and improvements from the EMSA analysis framework (including the correct Formulation for Independent Sites and the **Tight Binding / Morrison Equation** to handle ligand depletion effects).

In [3]:


def model_single(L, rf, rb, kd):
    """Standard 1:1 Langmuir binding isotherm"""
    return rf + (rb - rf) * (L / (kd + L))

def model_hill(L, rf, rb, kd, n):
    """Cooperative binding model (Hill equation)"""
    # Using a small epsilon to prevent 0 raised to a negative power if n can go negative
    return rf + (rb - rf) * (L**n / (kd**n + L**n))

def model_independent(L, rf, rb, kd1, kd2):
    """Two independent, non-interacting binding sites"""
    return rf + (rb - rf) * 0.5 * ((L / (kd1 + L)) + (L / (kd2 + L)))


def model_independent_biphasic(L, rf, rb1, rb2, kd1, kd2):
    """Two independent sites with distinct signal outputs """
    return rf + (rb1 - rf) * (L / (kd1 + L)) + (rb2 - rf) * (L / (kd2 + L))

def model_independent_biphasic2(L, rf, rb1, rb2, kd1, kd2,n,m):
    """Two independent sites with distinct signal outputs coaparative """
    return rf + (rb1 - rf) * (L**n / (kd1**n + L**n)) + (rb2 - rf) * (L**m / (kd2**m + L**m))


def model_sequential(L, rP, rPL, rPL2, k1, k2):
    """Two-site sequential binding model (Adair formulation)"""
    denom = 1 + (L / k1) + (L**2 / (k1 * k2))
    return (rP + rPL*(L/k1) + rPL2*(L**2/(k1*k2))) / denom

def model_tight(L, rf, rb, kd, probe):
    """Tight Binding (Morrison Equation) - accounts for ligand depletion"""
    term = L + probe + kd
    # np.clip or safeguarding the sqrt prevents NaN runtime warnings during optimization
    sqrt_term = np.sqrt(np.clip(term**2 - 4 * L * probe, 0, None))
    fraction = (term - sqrt_term) / (2 * probe)
    return rf + (rb - rf) * fraction


r_min = float(np.min(y_data))
r_max_obs = float(np.max(y_data))

# Dynamic intermediate guess for biphasic transitions
r_mid = float(r_min + 0.4 * (r_max_obs - r_min))

models = {
    "Single-Site": (
        model_single,
        [r_min, r_max_obs, 50.0],
        ([0.0, r_max_obs * 0.7, 1e-3], [0.45, 0.45, 10000.0]),
        ['rf', 'rb', 'kd']
    ),
    "Hill": (
        model_hill,
        [r_min, r_max_obs, 50.0, 1.0],
        ([0.0, r_max_obs * 0.7, 1e-3, 0.1], [0.45, 0.45, 10000.0, 5.0]),  # Fixed: Now contains 4 elements
        ['rf', 'rb', 'kd', 'n']
    ),
    "Independent (Biphasic)": (
        model_independent_biphasic,
        [r_min, r_mid, r_max_obs, 25.0, 4000.0],
        ([0.0, r_min, r_mid, 1e-3, 1e-3], [0.45, r_mid * 1.3, 0.45, 500.0, 100000.0]),
        ['rf', 'rb1', 'rb2', 'kd1', 'kd2']
    ),
     "Independent (Biphasic2)": (
        model_independent_biphasic2,
        [r_min, r_mid, r_max_obs, 25.0, 4000.0,1.0,1.0],
        ([0.0, r_min, r_mid, 1e-3, 1e-3,0.1,0.1], [0.45, r_mid * 1.3, 0.45, 500.0, 100000.0,5.0,5.0]),
        ['rf', 'rb1', 'rb2', 'kd1', 'kd2','n','m']
    ),
    "Independent": (
        model_independent,
        [r_min, r_max_obs, 25.0, 4000.0],
        ([0.0, r_max_obs * 0.7, 1e-3, 1e-3], [0.45, 0.45, 500.0, 50000.0]),
        ['rf', 'rb', 'kd1', 'kd2']
    ),
    "Sequential": (
        model_sequential,
        [r_min, r_mid, r_max_obs, 25.0, 4000.0],
        ([0.0, 0.0, 0.0, 1e-3, 1e-3], [0.45, 0.45, 0.45, 500.0, 100000.0]),
        ['rP', 'rPL', 'rPL2', 'k1', 'k2']
    ),
    "Tight Binding": (
        model_tight,
        [r_min, r_max_obs, 10.0, 20.0],
        ([0.0, r_max_obs * 0.7, 1e-3, 0.1], [0.45, 0.45, 5000.0, 500.0]),
        ['rf', 'rb', 'kd', 'probe_conc']
    )
}



## 2. Model Selection Filter
Choose which models you want to include in the statistical comparison.

In [4]:
selected_models = [ "Single-Site","Hill","Independent","Sequential","Tight Binding","Independent (Biphasic2)"]
models_to_fit = {k: v for k, v in models.items() if k in selected_models}
print("Models selected for evaluation:", list(models_to_fit.keys()))

Models selected for evaluation: ['Single-Site', 'Hill', 'Independent (Biphasic2)', 'Independent', 'Sequential', 'Tight Binding']


## 4. Curve Fitting and Monte Carlo Simulation (with AICc)
Fits each selected model, calculates **AICc** to score model performance while penalizing complexity, and runs 1000 Monte Carlo loops to evaluate parameter error spreads.

In [ ]:
stats = {}
mc_samples = {}
results_list = []

np.random.seed(42)

for name, (func, p0, bnds, p_names) in models_to_fit.items():
    try:
        # --- 1. Best Fit ---
        popt, pcov = curve_fit(
            func, x_data, y_data,
            p0=p0,
            bounds=bnds,
            maxfev=20000
        )

        y_pred = func(x_data, *popt)
        residuals = y_data - y_pred
        
        sse = np.sum(residuals**2)
        n, k = len(y_data), len(popt)
        
        # AIC / AICc formulation
        if sse > 0:
            aic = n * np.log(sse/n) + 2*k
            if n > (k + 1):
                aicc = aic + (2 * k**2 + 2 * k) / (n - k - 1)
            else:
                aicc = np.inf
        else:
            aic = aicc = -np.inf

        residual_sd = np.sqrt(sse / (n - k)) 

        # --- 2. Monte Carlo Uncertainty Estimation ---
        samples = []
        n_iterations = 100000
        
        for _ in range(n_iterations):
            y_syn = y_pred + np.random.normal(0, residual_sd, size=len(y_data))
            try:
                psamp, _ = curve_fit(
                    func, x_data, y_syn,
                    p0=popt,
                    bounds=bnds,
                    maxfev=10000
                )
                samples.append(psamp)
            except:
                continue

        df_mc = pd.DataFrame(samples, columns=p_names)
        if not df_mc.empty:
            df_mc = df_mc.replace([np.inf, -np.inf], np.nan).dropna()
            df_mc = df_mc[(np.abs(df_mc - df_mc.mean()) <= 3 * df_mc.std()).all(axis=1)]

        stats[name] = {'AICc': aicc, 'AIC': aic, 'SSE': sse, 'popt': popt}
        mc_samples[name] = df_mc

        for p in p_names:
            results_list.append({
                'Model': name,
                'AICc': aicc,
                'Parameter': p,
                'Mean_Value': df_mc[p].mean() if not df_mc.empty else np.nan,
                'SD_Error': df_mc[p].std() if not df_mc.empty else np.nan
            })

    except Exception as e:
        print(f"Model {name} failed to fit: {e}")

# --- Display Model Comparison Table ---
if stats:
    best_model_name = min(stats, key=lambda x: stats[x]['AICc'])
    print("\n--- MODEL SELECTION SUMMARY ---")
    print(f"{'Model':<15} | {'AICc':>8} | {'AIC':>8} | {'SSE':>8}")
    print("-" * 48)
    for name, val in stats.items():
        print(f"{name:15} | {val['AICc']:8.2f} | {val['AIC']:8.2f} | {val['SSE']:8.2f}")

    print(f"\nBest model (lowest AICc): {best_model_name}")
else:
    print("Error: No models converged successfully.")

## 5. Plot Fits and Residuals
Generates a multi-panel plot visualizing the experimental dataset against the mathematical fits and corresponding residual profiles.

In [ ]:
if stats:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    
    # Generate fine x scale for smooth plotting
    x_fit = np.logspace(np.log10(max(0.01, x_data[1]*0.1)), np.log10(x_data[-1]), 300)
    
    for name, val in stats.items():
        func = models[name][0]
        ax1.plot(x_fit, func(x_fit, *val['popt']), label=name, lw=2)
        ax2.scatter(x_data, y_data - func(x_data, *val['popt']), label=name, alpha=0.7)

    ax1.errorbar(x_data, y_data, fmt='o', color='black', capsize=3, zorder=10, label='Data')
    ax1.set_xscale('log')
    ax1.set_ylabel("Anisotropy (mP)")
    ax1.set_title("Fluorescence Anisotropy Model Fitting Comparison")
    ax1.legend()
    # --- HERE IS THE MODIFICATION ---
    ax1.set_ylim(0, np.max(y_data)*1.2)
    
    ax2.axhline(0, color='red', linestyle='--', lw=1.2)
    ax2.set_ylabel("Residuals")
    ax2.set_xlabel("[Ligand]")
    
    plt.tight_layout()
    plt.savefig("anisotropy_models_fit.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
if stats:
    num_models = len(stats)
    # Creates a dedicated multi-panel matrix: each model gets its own row (Fit + Residuals)
    fig, axes = plt.subplots(
        num_models, 2, 
        figsize=(12, 3.5 * num_models), 
        gridspec_kw={'width_ratios': [3, 1]}, 
        squeeze=False
    )
    
    # Generate fine x scale for smooth plotting across log space
    x_fit = np.logspace(np.log10(max(0.01, x_data[1]*0.1)), np.log10(x_data[-1]), 300)
    
    for idx, (name, val) in enumerate(stats.items()):
        func = models[name][0]
        ax_fit = axes[idx, 0]
        ax_res = axes[idx, 1]
        
        # 1. Independent Main Fit Plotting
        ax_fit.plot(x_data, y_data, 'o', color='black', zorder=10, label='Data')
        ax_fit.plot(x_fit, func(x_fit, *val['popt']), color='crimson', lw=2, label='Model Fit')
        
        ax_fit.set_xscale('log')
        ax_fit.set_ylabel("Anisotropy (mP)", fontsize=10)
        ax_fit.set_title(f"{name}", fontsize=11, fontweight='bold', pad=4)
        ax_fit.set_ylim(0, np.max(y_data) * 1.2)
        ax_fit.grid(True, which="both", ls="--", alpha=0.3)
        ax_fit.legend(loc='upper left', fontsize=8.5, frameon=True)
        
        # 2. Independent Residual Plotting
        residuals = y_data - func(x_data, *val['popt'])
        ax_res.scatter(x_data, residuals, color='royalblue', edgecolor='midnightblue', alpha=0.8, s=20, zorder=5)
        ax_res.axhline(0, color='red', linestyle='--', lw=1.2)
        
        ax_res.set_xscale('log')
        ax_res.set_ylabel("Residuals", fontsize=9)
        ax_res.set_xlabel("[Ligand]", fontsize=10)
        ax_res.set_title(f"{name} Residuals", fontsize=10, pad=4)
        ax_res.grid(True, which="both", ls="--", alpha=0.3)
        
        # Scale the y-axis bound for the residual window dynamically to avoid distortion
        res_bound = max(abs(np.min(residuals)), abs(np.max(residuals))) * 1.3
        ax_res.set_ylim(-res_bound, res_bound)
        
    plt.tight_layout()
    plt.savefig("anisotropy_models_collage.png", dpi=300, bbox_inches='tight')

## 6. Parameter Error Distributions (Corner Plot)
Visualizes the Monte Carlo parameter covariance matrix and error histograms for the best-performing model.

In [ ]:
if stats and not mc_samples[best_model_name].empty:
    df_best = mc_samples[best_model_name]
    try:
        g = sns.pairplot(df_best, corner=True, kind='kde', diag_kind='kde', plot_kws={'fill': True, 'alpha': 0.4})
    except:
        g = sns.pairplot(df_best, corner=True, kind='scatter', diag_kind='hist', plot_kws={'alpha': 0.5})
        
    g.fig.suptitle(f"Parameter Error Distributions & Correlations ({best_model_name})", y=1.02)
    g.fig.savefig("anisotropy_corner_plot.png", dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No MC samples available to produce corner plot.")

## 7. Export Fit Summaries
Saves the calculated parameter means, standard deviations, and AICc values into a neat CSV file.

In [ ]:
if results_list:
    results_df = pd.DataFrame(results_list)
    results_df.to_csv("anisotropy_binding_results.csv", index=False)
    print("Successfully exported results summary to 'anisotropy_binding_results.csv'.")
    print(results_df.head(20))
else:
    print("No results to export.")

In [ ]:
if stats:
    best_func = models[best_model_name][0]
    best_popt = stats[best_model_name]['popt']

    x_fit = np.logspace(np.log10(max(0.01, x_data[1]*0.1)), np.log10(x_data[-1]), 300)
    y_fit = best_func(x_fit, *best_popt)

    y_pred = best_func(x_data, *best_popt)
    residuals = y_data - y_pred

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(5, 4.5),
        sharex=True,
        gridspec_kw={'height_ratios': [3, 1]}
    )
    # Fit
    ax1.plot(x_fit, y_fit, lw=2, label=f"{best_model_name} fit")
    ax1.errorbar(
        x_data, y_data, 
        fmt='o', color='black', capsize=3, label='Data', zorder=10
    )

    ax1.set_xscale('log')
    ax1.set_ylim(0, np.max(y_data)*1.2)  # Preserves the customized maximum Y-scale bound
    ax1.set_ylabel("Anisotropy (mP)")
    ax1.set_title(f"Best Fit Model: {best_model_name}")
    ax1.legend()
    
    # Residuals
    ax2.axhline(0, lw=1.2, color='red', linestyle='--')
    ax2.scatter(x_data, residuals, alpha=0.7)

    ax2.set_xlabel("[Ligand]")
    ax2.set_ylabel("Residuals")

    plt.tight_layout()
    plt.savefig("best_model_fit.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Best model plot saved as 'best_model_fit.png'")

In [11]:
stats

{'Single-Site': {'AICc': np.float64(-154.43715808519684),
  'AIC': np.float64(-156.15144379948256),
  'SSE': np.float64(0.002202765226997935),
  'popt': array([4.63909528e-02, 1.48332589e-01, 1.16190775e+03])},
 'Hill': {'AICc': np.float64(-164.04604512699223),
  'AIC': np.float64(-167.1229682039153),
  'SSE': np.float64(0.0010715146923377186),
  'popt': array([3.72718531e-02, 1.96391288e-01, 4.26090473e+03, 4.49570436e-01])},
 'Independent (Biphasic2)': {'AICc': np.float64(-207.5550864639474),
  'AIC': np.float64(-218.75508646394738),
  'SSE': np.float64(4.359943445147753e-05),
  'popt': array([4.42547264e-02, 9.63449453e-02, 1.03828215e-01, 1.60395771e+02,
         8.36152881e+03, 3.36014474e+00, 3.46182747e+00])},
 'Independent': {'AICc': np.float64(-169.4735668392281),
  'AIC': np.float64(-172.55048991615118),
  'SSE': np.float64(0.0007925848313541387),
  'popt': array([4.04550682e-02, 1.69931568e-01, 2.12688886e+02, 1.46497013e+04])},
 'Sequential': {'AICc': np.float64(-168.139201